# 40 · Unity Catalog metadata, tables and lineage

**Scenario:** Distinguish catalogs/schemas/tables/volumes and inspect course table metadata.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **delta**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Workspace setup — Choose course-owned output names

Set `target_catalog` to a catalog where you can create a schema. This lab creates a per-user course schema and uses lab-specific names. Rerunning a write lab can replace its own course tables. Nothing is written to the `samples` catalog.


In [ ]:
LAB='40'

import hashlib, re
dbutils.widgets.text('target_catalog','')
target_catalog=dbutils.widgets.get('target_catalog').strip() or spark.sql('SELECT current_catalog()').first()[0]
assert target_catalog!='samples', 'Set target_catalog to a writable catalog, not the read-only samples catalog.'
user=spark.sql('SELECT current_user()').first()[0]
schema='pyspark_lab_'+hashlib.sha256(user.encode()).hexdigest()[:10]
def quote_identifier(value): return '`'+value.replace('`','``')+'`'
namespace=f'{quote_identifier(target_catalog)}.{quote_identifier(schema)}'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {namespace}')
def table(name):
    assert re.fullmatch(r'[a-z][a-z0-9_]*',name)
    return f'{namespace}.`lab_{LAB}_{name}`'
print('Course-only namespace:',namespace)


## Step 2 — Persist source provenance

The source asset is read-only. Course outputs live in a writable per-user schema.


In [ ]:
target=table('governed')
df.withColumn('_source_asset',F.lit(SOURCE_ASSET)).write.format('delta').mode('overwrite').saveAsTable(target)
spark.sql(f"COMMENT ON TABLE {target} IS 'PySpark course output derived from samples.nyctaxi.trips'")
spark.sql(f'DESCRIBE TABLE EXTENDED {target}').show(30,truncate=False)


**Expected check:** The output includes a source-asset field and table comment.


## Step 3 — Read information_schema using bound values

Information schema exposes metadata subject to permissions. Actual lineage can also be inspected in Catalog Explorer; do not assume access to system lineage tables.


In [ ]:
metadata=spark.sql(f"""SELECT table_catalog, table_schema, table_name, table_type
FROM {quote_identifier(target_catalog)}.information_schema.tables
WHERE table_schema=:schema_name AND table_name=:table_name""",args={'schema_name':schema,'table_name':f'lab_{LAB}_governed'})
assert metadata.count()==1
metadata.show(truncate=False)
print('No grants or ownership changes are made by this course.')


**Expected check:** Metadata identifies the course table without modifying access rules.


## Practice

Inspect upstream/downstream lineage in Catalog Explorer. List the least privileges needed to read the asset and write your target.

## Solution guide

Use appropriate catalog/schema usage and table read/create permissions. A Unity Catalog volume is file storage; a managed table is governed structured data with its own lifecycle.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
